# 15 GMM and Spectral on D2, D3, D4 and D8

The report compares six methods on every dataset. Notebooks 08, 11 and 13 tune K-Means, DBSCAN and
HDBSCAN on D2 (earthquakes 2023), D8 (HDB resale), D3 (Uber) and D4 (Ecoli); this notebook adds GMM and
Spectral to those four, in the report's shared columns.

- Both use **the K that K-Means uses on the same dataset**, read from the earlier notebooks' results, so
  the methods differ only in how they cluster. GMM's covariance type is chosen by BIC at that K
  (`tuning.select_gmm_by_bic`); on D4 the best-ARI K of K-Means gets a row too.
- Spectral uses a 10-nearest-neighbour graph (sparse), which fits in memory at 15,000 and 20,000 points.
- Each dataset is scored exactly as in its own notebook: the same features, silhouette sample, seed and
  fake data (shuffled columns or coordinates, as for K-Means).

Run after 08, 11 and 13. Writes `results/comparison/gmm_spectral.csv`.

In [ ]:
import json
import re
import time
import warnings

import numpy as np
import pandas as pd

from sc4020 import methods, tuning
from sc4020.datasets import load_dataset
from sc4020.datasets.download_dataset import latest_datagov
from sc4020.datasets.preprocess_dataset import (
    hdb_features,
    price_index_series,
    standardize,
    to_radians,
    to_unit_vectors,
)
from sc4020.metrics import evaluate
from sc4020.paths import results_dir
from sc4020.tuning import null_gap, shuffle_columns_independently

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

SILHOUETTE_SAMPLE = 5000
N_NEIGHBOURS = 10
COLUMNS = [
    "n_clusters",
    "noise_fraction",
    "largest_cluster_share",
    "silhouette",
    "davies_bouldin",
    "ari",
    "nmi",
]
rows = []

In [ ]:
def gmm_at_k(X, k, seed):
    """GMM with k components and the covariance type of lowest BIC at that k."""
    selection = tuning.select_gmm_by_bic(X, k=k, k_range=[k], random_state=seed)
    return methods.gmm(random_state=seed, **selection.params), selection.params


def spectral_at_k(k, seed):
    """Spectral clustering on a sparse nearest-neighbour graph with k clusters."""
    return methods.spectral(k, affinity="nearest_neighbors", random_state=seed), {
        "n_clusters": k,
        "affinity": "nearest_neighbors",
    }


def k_from_setting(setting):
    """The K in a setting string such as 'K=6, n_init=1'."""
    return int(re.search(r"K=(\d+)", setting).group(1))


def add_row(dataset, method, choice, params, labels, seconds, scores, null_silhouette):
    rows.append(
        {
            "dataset": dataset,
            "method": method,
            "choice": choice,
            "params": json.dumps(params),
            **{c: scores.get(c) for c in COLUMNS},
            "fit_seconds": seconds,
            "null_silhouette": null_silhouette,
        }
    )

## D2 earthquakes 2023

As in notebook 08: models fit on 3D unit vectors, silhouette by great-circle distance on a 5,000-point
sample (seed 0), Davies-Bouldin on unit vectors, fake data = the degree columns shuffled (seed 0).

In [ ]:
SEED_D2 = 0
events = load_dataset("earthquakes_2023").events
degrees = events[["latitude", "longitude"]].to_numpy()
R, U = to_radians(degrees[:, 0], degrees[:, 1]), to_unit_vectors(degrees[:, 0], degrees[:, 1])
null_degrees = shuffle_columns_independently(degrees, np.random.default_rng(SEED_D2))
R_null = to_radians(null_degrees[:, 0], null_degrees[:, 1])
U_null = to_unit_vectors(null_degrees[:, 0], null_degrees[:, 1])

comparison = pd.read_csv(results_dir("earthquakes_2023") / "method_comparison.csv")
K_D2 = k_from_setting(comparison.loc[comparison["method"] == "K-means++", "setting"].iloc[0])
print(f"{len(U):,} earthquakes, K={K_D2}")

for method, (model, params) in {
    "GMM": gmm_at_k(U, K_D2, SEED_D2),
    "Spectral": spectral_at_k(K_D2, SEED_D2),
}.items():
    labels, seconds = methods.fit_predict_timed(model, U)
    score = lambda radians, units, lab: evaluate(
        radians, lab, silhouette_sample=SILHOUETTE_SAMPLE, seed=SEED_D2,
        metric="haversine", X_davies_bouldin=units,
    )
    null = score(R_null, U_null, model.fit_predict(U_null))
    add_row("earthquakes_2023", method, "rule", params, labels, seconds, score(R, U, labels), null["silhouette"])

## D8 HDB resale

As in notebook 11: the 20,000-sale sample (seed 0), recorded values standardized, ARI against flat
type, silhouette on 5,000 points, fake data = columns shuffled (seed 0).

In [ ]:
SEED_D8 = 0
resale = load_dataset("hdb_resale", n_samples=None, jitter=False).all_sales
all_features = hdb_features(resale, price_index_series(pd.read_csv(latest_datagov("price_index"))))
sample = all_features.sample(n=20_000, random_state=SEED_D8)
flat_type = resale.loc[sample.index, "flat_type"].to_numpy()
X_d8 = standardize(sample)
X_d8_null = shuffle_columns_independently(X_d8, np.random.default_rng(SEED_D8))

comparison = pd.read_csv(results_dir("hdb_resale") / "comparison.csv")
K_D8 = k_from_setting(comparison.loc[comparison["method"] == "K-means++", "setting"].iloc[0])
print(f"{len(all_features):,} sales, sample {len(sample):,}, K={K_D8}")

for method, (model, params) in {
    "GMM": gmm_at_k(X_d8, K_D8, SEED_D8),
    "Spectral": spectral_at_k(K_D8, SEED_D8),
}.items():
    labels, seconds = methods.fit_predict_timed(model, X_d8)
    scores = evaluate(X_d8, labels, y_true=flat_type, silhouette_sample=SILHOUETTE_SAMPLE, seed=SEED_D8)
    null = evaluate(X_d8_null, model.fit_predict(X_d8_null), silhouette_sample=SILHOUETTE_SAMPLE, seed=SEED_D8)
    add_row("hdb_resale", method, "rule", params, labels, seconds, scores, null["silhouette"])

## D3 Uber and D4 Ecoli

As in notebook 13 (shared table): K from `chosen_params.json`, scores with seed 42, fake-data silhouette
as the mean over 3 shuffled copies (`null_gap`). On D4 the best-ARI K of K-Means gets a row too.

In [ ]:
SEED_D3_D4 = 42
chosen = json.loads((results_dir("hotspot") / "chosen_params.json").read_text())
for ds in [load_dataset("uber_pickups"), load_dataset("ecoli")]:
    k_choice = chosen["k"][ds.name]
    ks = [("rule", k_choice["selected"])]
    if ds.target is not None:
        ks.append(("best_ari", k_choice["best_ari"]))
    print(f"{ds.name}: {len(ds.data):,} points, K {dict(ks)}")
    for choice, k in ks:
        for method, (model, params) in {
            "GMM": gmm_at_k(ds.data, k, SEED_D3_D4),
            "Spectral": spectral_at_k(k, SEED_D3_D4),
        }.items():
            labels, seconds = methods.fit_predict_timed(model, ds.data)
            scores = evaluate(ds.data, labels, ds.target, seed=SEED_D3_D4)
            fit = lambda X, model=model: model.fit_predict(X)
            gap = null_gap(ds.data, fit, null="shuffled", n_nulls=3, random_state=SEED_D3_D4)
            add_row(ds.name, method, choice, params, labels, seconds, scores, gap["null_silhouette"])

In [ ]:
table = pd.DataFrame(rows)
table.to_csv(results_dir("comparison") / "gmm_spectral.csv", index=False)
table.round(3)

## D1 make_moons in the shared columns

D1 has no shared-table rows yet (notebook 03 writes its own results only). `run.shared_table_rows`
gives them the same way notebook 01 does for D6: the four tuned methods, Agglomerative and Spectral at
the number of classes, with the fake-data silhouette. Written to `results/comparison/report_d1.csv`.

In [ ]:
from sc4020 import run

d1_rows, _ = run.shared_table_rows(load_dataset("make_moons"))
d1_table = pd.DataFrame(d1_rows)
d1_table.to_csv(results_dir("comparison") / "report_d1.csv", index=False)
d1_table.drop(columns=["params"]).round(3)